In [ ]:
# pip install pandas openpyxl

In [ ]:
import pandas as pd
import os
import glob
from pathlib import Path

In [ ]:
# ====================== 配置项（修改为实际路径） ======================
BASE_DIR = r"C:\Users\33759\Desktop\陈正扬\初始数据"
OUTPUT_DIR_WITH_SUBJECT = r"C:\Users\33759\Desktop\陈正扬\处理后数据（含学科）"
YEARS = ["2020", "2021", "2022", "2023", "2024", "2025"]
IGNORE_FILES = ["高校辅导员研究.xlsx", "中国特色社会主义理论体系.xlsx"]
SPECIAL_PROJECT_MAP = {
    "高校辅导员研究.xlsx": "专项项目（辅导员研究）",
    "中国特色社会主义理论体系.xlsx": "专项项目（中国特色社会主义）"
}

# ====================== 学科门类归一化函数 ======================
def normalize_subject(subject_name):
    if pd.isna(subject_name):
        return "未分类"

    # 去除换行符、空格
    subject_name = str(subject_name).strip().replace("\n", "").replace(" ", "")

    if "交叉学科" in subject_name or "综合研究" in subject_name:
        return "交叉学科/综合研究"
    if "马克思主义" in subject_name or "思想政治教育" in subject_name:
        return "马克思主义/思想政治教育"

    subject_mapping = {
        "图书馆、情报": "图书馆、情报与文献学",
        "图书馆、情报与文献": "图书馆、情报与文献学",
        "教育学": "教育学/心理学",
        "心理学": "教育学/心理学",
        "经济学": "经济学/管理学",
        "管理学": "经济学/管理学",
        "文学": "文学/语言学",
        "语言学": "文学/语言学",
        "图书馆、情报与文献学": "图书馆、情报与文献学"
    }
    return subject_mapping.get(subject_name, subject_name)

# ====================== 通用函数======================
def read_excel_auto_header(file_path):
    df_raw = pd.read_excel(file_path, header=None)
    header_row = None
    for idx, row in df_raw.iterrows():
        row_clean = " ".join([str(cell).strip().lower() for cell in row if pd.notna(cell)])
        if "项目名称" in row_clean and "学校名称" in row_clean:
            header_row = idx
            break
    if header_row is None:
        raise ValueError(f"文件 {file_path} 未找到表头行！")
    df = pd.read_excel(file_path, header=header_row)
    df.columns = [str(col).strip() for col in df.columns]
    return df

def clean_dataframe(df):
    df = df.drop_duplicates()
    df = df.dropna(subset=["项目名称", "学校名称"])
    df = df.fillna("")
    return df

def unify_column_names(df):
    col_mapping = {"学科": "学科门类"}
    df = df.rename(columns=col_mapping)
    return df

# ====================== 修正含学科数据处理函数 ======================
def process_data_with_subject():
    print("=== 开始处理【含学科】数据集（新增归一化） ===")
    all_dfs = []
    for year in YEARS:
        year_dir = os.path.join(BASE_DIR, year)
        if not os.path.exists(year_dir):
            print(f"警告：年份文件夹 {year_dir} 不存在，跳过")
            continue
        excel_files = glob.glob(os.path.join(year_dir, "*.xlsx"))
        for file_path in excel_files:
            file_name = os.path.basename(file_path)
            if file_name in IGNORE_FILES:
                print(f"跳过文件：{file_path}")
                continue
            try:
                df = read_excel_auto_header(file_path)
                df = unify_column_names(df)
                df["立项年份"] = year

                # 核心新增：对学科门类列做归一化（解决重复）
                if "学科门类" in df.columns:
                    df["学科门类"] = df["学科门类"].apply(normalize_subject)

                all_dfs.append(df)
                print(f"成功处理：{file_path}")
            except Exception as e:
                print(f"处理失败 {file_path}：{str(e)}")

    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        combined_df = clean_dataframe(combined_df)
        # 保存时保留所有学科（无过滤）
        output_path = os.path.join(OUTPUT_DIR_WITH_SUBJECT, "合并数据（含学科）.xlsx")
        combined_df.to_excel(output_path, index=False)
        # 打印所有学科列表，确认无重复
        all_subjects = combined_df["学科门类"].unique()
        print(f"✅ 所有学科门类（共{len(all_subjects)}个）：{all_subjects}")
        print(f"【含学科】数据集保存完成：{output_path}")
    else:
        print("警告：无【含学科】数据可处理！")



# ====================== 主执行 ======================
if __name__ == "__main__":
    Path(OUTPUT_DIR_WITH_SUBJECT).mkdir(parents=True, exist_ok=True)
    process_data_with_subject()
    print("\n=== 所有数据处理完成！ ===")